# 06 - Ternary Recovery Gemma 4

Notebook operacional para o próximo passo correto após o fracasso do snap ternário direto: ternary subset -> LoRA recovery -> eval. Esta versão evita salvar um modelo merged pesado no Drive durante a avaliação final.

In [ ]:
# Auto-bootstrap para Colab/local. Rode esta célula primeiro.
from pathlib import Path
import os
import sys
import json

IN_COLAB = 'google.colab' in sys.modules
PROJECT_NAME = 'gemma4_theory'
COLAB_DRIVE_ROOT = '/content/drive/MyDrive/RPT'
LOCAL_ROOT = str(Path.cwd().resolve())

if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)
    PROJECT_ROOT = Path(COLAB_DRIVE_ROOT) / PROJECT_NAME
    PROJECT_ROOT.mkdir(parents=True, exist_ok=True)
    !pip -q install transformers accelerate sentencepiece safetensors peft datasets bitsandbytes
else:
    PROJECT_ROOT = Path(LOCAL_ROOT) / PROJECT_NAME if Path(LOCAL_ROOT).name != PROJECT_NAME else Path(LOCAL_ROOT)
    PROJECT_ROOT.mkdir(parents=True, exist_ok=True)

DATA_DIR = PROJECT_ROOT / 'data'
UTILS_DIR = PROJECT_ROOT / 'utils'
RESULTS_DIR = PROJECT_ROOT / 'results'
CHECKPOINTS_DIR = PROJECT_ROOT / 'checkpoints'
for p in [DATA_DIR, UTILS_DIR, RESULTS_DIR, CHECKPOINTS_DIR]:
    p.mkdir(parents=True, exist_ok=True)

questions_payload = '[\n  {\n    "id": 1,\n    "question": "Qual a capital da França?",\n    "expected_answer": "Paris",\n    "accepted_answers": [\n      "Paris"\n    ],\n    "category": "factual",\n    "notes": ""\n  },\n  {\n    "id": 2,\n    "question": "Qual a capital da Alemanha?",\n    "expected_answer": "Berlim",\n    "accepted_answers": [\n      "Berlim",\n      "Berlin"\n    ],\n    "category": "factual",\n    "notes": ""\n  },\n  {\n    "id": 3,\n    "question": "Qual a capital da Itália?",\n    "expected_answer": "Roma",\n    "accepted_answers": [\n      "Roma",\n      "Rome"\n    ],\n    "category": "factual",\n    "notes": ""\n  },\n  {\n    "id": 4,\n    "question": "Qual a capital da Espanha?",\n    "expected_answer": "Madri",\n    "accepted_answers": [\n      "Madri",\n      "Madrid"\n    ],\n    "category": "factual",\n    "notes": ""\n  },\n  {\n    "id": 5,\n    "question": "Qual a capital de Portugal?",\n    "expected_answer": "Lisboa",\n    "accepted_answers": [\n      "Lisboa"\n    ],\n    "category": "factual",\n    "notes": ""\n  },\n  {\n    "id": 6,\n    "question": "Qual a capital do Japão?",\n    "expected_answer": "Tóquio",\n    "accepted_answers": [\n      "Tóquio",\n      "Toquio",\n      "Tokyo"\n    ],\n    "category": "factual",\n    "notes": ""\n  },\n  {\n    "id": 7,\n    "question": "Qual a capital da China?",\n    "expected_answer": "Pequim",\n    "accepted_answers": [\n      "Pequim",\n      "Beijing",\n      "Pekin"\n    ],\n    "category": "factual",\n    "notes": ""\n  },\n  {\n    "id": 8,\n    "question": "Qual a capital da Argentina?",\n    "expected_answer": "Buenos Aires",\n    "accepted_answers": [\n      "Buenos Aires"\n    ],\n    "category": "factual",\n    "notes": ""\n  },\n  {\n    "id": 9,\n    "question": "Qual a capital do Canadá?",\n    "expected_answer": "Ottawa",\n    "accepted_answers": [\n      "Ottawa"\n    ],\n    "category": "factual",\n    "notes": ""\n  },\n  {\n    "id": 10,\n    "question": "Qual a capital da Austrália?",\n    "expected_answer": "Camberra",\n    "accepted_answers": [\n      "Camberra",\n      "Canberra"\n    ],\n    "category": "factual",\n    "notes": ""\n  },\n  {\n    "id": 11,\n    "question": "Quem pintou a Mona Lisa?",\n    "expected_answer": "Leonardo da Vinci",\n    "accepted_answers": [\n      "Leonardo da Vinci",\n      "Da Vinci"\n    ],\n    "category": "factual",\n    "notes": ""\n  },\n  {\n    "id": 12,\n    "question": "Quem escreveu Dom Quixote?",\n    "expected_answer": "Miguel de Cervantes",\n    "accepted_answers": [\n      "Miguel de Cervantes",\n      "Cervantes"\n    ],\n    "category": "factual",\n    "notes": ""\n  },\n  {\n    "id": 13,\n    "question": "Quem escreveu Hamlet?",\n    "expected_answer": "William Shakespeare",\n    "accepted_answers": [\n      "William Shakespeare",\n      "Shakespeare"\n    ],\n    "category": "factual",\n    "notes": ""\n  },\n  {\n    "id": 14,\n    "question": "Quem descobriu a penicilina?",\n    "expected_answer": "Alexander Fleming",\n    "accepted_answers": [\n      "Alexander Fleming",\n      "Fleming"\n    ],\n    "category": "factual",\n    "notes": ""\n  },\n  {\n    "id": 15,\n    "question": "Quem desenvolveu a teoria da relatividade?",\n    "expected_answer": "Albert Einstein",\n    "accepted_answers": [\n      "Albert Einstein",\n      "Einstein"\n    ],\n    "category": "factual",\n    "notes": ""\n  },\n  {\n    "id": 16,\n    "question": "Qual planeta é conhecido como Planeta Vermelho?",\n    "expected_answer": "Marte",\n    "accepted_answers": [\n      "Marte",\n      "Mars"\n    ],\n    "category": "factual",\n    "notes": ""\n  },\n  {\n    "id": 17,\n    "question": "Qual é o maior planeta do Sistema Solar?",\n    "expected_answer": "Júpiter",\n    "accepted_answers": [\n      "Júpiter",\n      "Jupiter"\n    ],\n    "category": "factual",\n    "notes": ""\n  },\n  {\n    "id": 18,\n    "question": "Quantos continentes existem tradicionalmente?",\n    "expected_answer": "7",\n    "accepted_answers": [\n      "7",\n      "sete"\n    ],\n    "category": "factual",\n    "notes": ""\n  },\n  {\n    "id": 19,\n    "question": "Qual é o oceano mais extenso da Terra?",\n    "expected_answer": "Oceano Pacífico",\n    "accepted_answers": [\n      "Oceano Pacífico",\n      "Pacífico",\n      "Pacific Ocean"\n    ],\n    "category": "factual",\n    "notes": ""\n  },\n  {\n    "id": 20,\n    "question": "Qual é o metal cujo símbolo químico é Au?",\n    "expected_answer": "Ouro",\n    "accepted_answers": [\n      "Ouro",\n      "Gold"\n    ],\n    "category": "factual",\n    "notes": ""\n  },\n  {\n    "id": 21,\n    "question": "Qual é o resultado de 12 x 8?",\n    "expected_answer": "96",\n    "accepted_answers": [\n      "96"\n    ],\n    "category": "math",\n    "notes": ""\n  },\n  {\n    "id": 22,\n    "question": "Quanto é 144 dividido por 12?",\n    "expected_answer": "12",\n    "accepted_answers": [\n      "12"\n    ],\n    "category": "math",\n    "notes": ""\n  },\n  {\n    "id": 23,\n    "question": "Quanto é 15 ao quadrado?",\n    "expected_answer": "225",\n    "accepted_answers": [\n      "225"\n    ],\n    "category": "math",\n    "notes": ""\n  },\n  {\n    "id": 24,\n    "question": "Qual é a raiz quadrada de 81?",\n    "expected_answer": "9",\n    "accepted_answers": [\n      "9"\n    ],\n    "category": "math",\n    "notes": ""\n  },\n  {\n    "id": 25,\n    "question": "Quanto é 7 + 18?",\n    "expected_answer": "25",\n    "accepted_answers": [\n      "25"\n    ],\n    "category": "math",\n    "notes": ""\n  },\n  {\n    "id": 26,\n    "question": "Quanto é 100 - 37?",\n    "expected_answer": "63",\n    "accepted_answers": [\n      "63"\n    ],\n    "category": "math",\n    "notes": ""\n  },\n  {\n    "id": 27,\n    "question": "Quanto é 9 x 9?",\n    "expected_answer": "81",\n    "accepted_answers": [\n      "81"\n    ],\n    "category": "math",\n    "notes": ""\n  },\n  {\n    "id": 28,\n    "question": "Quanto é 5! (fatorial de 5)?",\n    "expected_answer": "120",\n    "accepted_answers": [\n      "120"\n    ],\n    "category": "math",\n    "notes": ""\n  },\n  {\n    "id": 29,\n    "question": "Qual é o próximo número na sequência 2, 4, 8, 16, ...?",\n    "expected_answer": "32",\n    "accepted_answers": [\n      "32"\n    ],\n    "category": "math",\n    "notes": ""\n  },\n  {\n    "id": 30,\n    "question": "Se um triângulo tem lados 3, 4 e 5, ele é retângulo?",\n    "expected_answer": "Sim",\n    "accepted_answers": [\n      "Sim",\n      "sim",\n      "Yes"\n    ],\n    "category": "math",\n    "notes": ""\n  },\n  {\n    "id": 31,\n    "question": "A água ferve a aproximadamente quantos graus Celsius ao nível do mar?",\n    "expected_answer": "100",\n    "accepted_answers": [\n      "100",\n      "100°C",\n      "100 graus Celsius"\n    ],\n    "category": "science",\n    "notes": ""\n  },\n  {\n    "id": 32,\n    "question": "Qual gás as plantas absorvem principalmente para fazer fotossíntese?",\n    "expected_answer": "Dióxido de carbono",\n    "accepted_answers": [\n      "Dióxido de carbono",\n      "CO2",\n      "Gás carbônico"\n    ],\n    "category": "science",\n    "notes": ""\n  },\n  {\n    "id": 33,\n    "question": "Qual é a fórmula química da água?",\n    "expected_answer": "H2O",\n    "accepted_answers": [\n      "H2O"\n    ],\n    "category": "science",\n    "notes": ""\n  },\n  {\n    "id": 34,\n    "question": "Qual partícula tem carga elétrica negativa?",\n    "expected_answer": "Elétron",\n    "accepted_answers": [\n      "Elétron",\n      "Eletron",\n      "Electron"\n    ],\n    "category": "science",\n    "notes": ""\n  },\n  {\n    "id": 35,\n    "question": "Qual órgão do corpo humano bombeia sangue?",\n    "expected_answer": "Coração",\n    "accepted_answers": [\n      "Coração",\n      "Coracao"\n    ],\n    "category": "science",\n    "notes": ""\n  },\n  {\n    "id": 36,\n    "question": "Qual é o satélite natural da Terra?",\n    "expected_answer": "Lua",\n    "accepted_answers": [\n      "Lua",\n      "Moon"\n    ],\n    "category": "science",\n    "notes": ""\n  },\n  {\n    "id": 37,\n    "question": "Qual é o processo pelo qual um líquido vira vapor?",\n    "expected_answer": "Evaporação",\n    "accepted_answers": [\n      "Evaporação",\n      "Evaporacao"\n    ],\n    "category": "science",\n    "notes": ""\n  },\n  {\n    "id": 38,\n    "question": "Quantos cromossomos tem um ser humano típico em células somáticas?",\n    "expected_answer": "46",\n    "accepted_answers": [\n      "46"\n    ],\n    "category": "science",\n    "notes": ""\n  },\n  {\n    "id": 39,\n    "question": "Qual é a camada da atmosfera onde vivemos?",\n    "expected_answer": "Troposfera",\n    "accepted_answers": [\n      "Troposfera"\n    ],\n    "category": "science",\n    "notes": ""\n  },\n  {\n    "id": 40,\n    "question": "O DNA tem forma clássica de qual estrutura?",\n    "expected_answer": "Dupla hélice",\n    "accepted_answers": [\n      "Dupla hélice",\n      "Dupla helix",\n      "Double helix"\n    ],\n    "category": "science",\n    "notes": ""\n  },\n  {\n    "id": 41,\n    "question": "Em que ano o homem pisou na Lua pela primeira vez?",\n    "expected_answer": "1969",\n    "accepted_answers": [\n      "1969"\n    ],\n    "category": "history",\n    "notes": ""\n  },\n  {\n    "id": 42,\n    "question": "Quem foi o primeiro presidente do Brasil?",\n    "expected_answer": "Deodoro da Fonseca",\n    "accepted_answers": [\n      "Deodoro da Fonseca"\n    ],\n    "category": "history",\n    "notes": ""\n  },\n  {\n    "id": 43,\n    "question": "Quem foi o líder sul-africano associado ao fim do apartheid e eleito presidente em 1994?",\n    "expected_answer": "Nelson Mandela",\n    "accepted_answers": [\n      "Nelson Mandela",\n      "Mandela"\n    ],\n    "category": "history",\n    "notes": ""\n  },\n  {\n    "id": 44,\n    "question": "Em que país começou a Revolução Industrial?",\n    "expected_answer": "Reino Unido",\n    "accepted_answers": [\n      "Reino Unido",\n      "Inglaterra",\n      "Grã-Bretanha",\n      "Great Britain"\n    ],\n    "category": "history",\n    "notes": ""\n  },\n  {\n    "id": 45,\n    "question": "Qual império construiu o Coliseu?",\n    "expected_answer": "Império Romano",\n    "accepted_answers": [\n      "Império Romano",\n      "Roman Empire",\n      "Roma"\n    ],\n    "category": "history",\n    "notes": ""\n  },\n  {\n    "id": 46,\n    "question": "Em que ano caiu o Muro de Berlim?",\n    "expected_answer": "1989",\n    "accepted_answers": [\n      "1989"\n    ],\n    "category": "history",\n    "notes": ""\n  },\n  {\n    "id": 47,\n    "question": "Quem liderou a independência da Índia com estratégia de não violência?",\n    "expected_answer": "Mahatma Gandhi",\n    "accepted_answers": [\n      "Mahatma Gandhi",\n      "Gandhi"\n    ],\n    "category": "history",\n    "notes": ""\n  },\n  {\n    "id": 48,\n    "question": "Qual civilização antiga construiu Machu Picchu?",\n    "expected_answer": "Inca",\n    "accepted_answers": [\n      "Inca",\n      "Império Inca"\n    ],\n    "category": "history",\n    "notes": ""\n  },\n  {\n    "id": 49,\n    "question": "Quem foi o imperador francês derrotado em Waterloo?",\n    "expected_answer": "Napoleão Bonaparte",\n    "accepted_answers": [\n      "Napoleão Bonaparte",\n      "Napoleon Bonaparte",\n      "Napoleão"\n    ],\n    "category": "history",\n    "notes": ""\n  },\n  {\n    "id": 50,\n    "question": "Em que continente fica o Egito?",\n    "expected_answer": "África",\n    "accepted_answers": [\n      "África",\n      "Africa"\n    ],\n    "category": "history",\n    "notes": ""\n  },\n  {\n    "id": 51,\n    "question": "Se todos os gatos são mamíferos e Mimi é um gato, Mimi é mamífero?",\n    "expected_answer": "Sim",\n    "accepted_answers": [\n      "Sim",\n      "sim",\n      "Yes"\n    ],\n    "category": "logic",\n    "notes": ""\n  },\n  {\n    "id": 52,\n    "question": "Se hoje é segunda-feira, que dia será depois de 3 dias?",\n    "expected_answer": "Quinta-feira",\n    "accepted_answers": [\n      "Quinta-feira",\n      "Quinta",\n      "Thursday"\n    ],\n    "category": "logic",\n    "notes": ""\n  },\n  {\n    "id": 53,\n    "question": "Qual palavra não pertence ao grupo: maçã, banana, cenoura, uva?",\n    "expected_answer": "Cenoura",\n    "accepted_answers": [\n      "Cenoura"\n    ],\n    "category": "logic",\n    "notes": ""\n  },\n  {\n    "id": 54,\n    "question": "Se João é mais alto que Maria e Maria é mais alta que Ana, quem é a mais alta?",\n    "expected_answer": "João",\n    "accepted_answers": [\n      "João",\n      "Joao"\n    ],\n    "category": "logic",\n    "notes": ""\n  },\n  {\n    "id": 55,\n    "question": "Um carro percorre 60 km em 1 hora. Em 3 horas, mantendo a velocidade, percorre quantos km?",\n    "expected_answer": "180",\n    "accepted_answers": [\n      "180"\n    ],\n    "category": "logic",\n    "notes": ""\n  },\n  {\n    "id": 56,\n    "question": "Se 2 canetas custam 10 reais, quanto custam 5 canetas ao mesmo preço unitário?",\n    "expected_answer": "25",\n    "accepted_answers": [\n      "25",\n      "25 reais",\n      "R$25"\n    ],\n    "category": "logic",\n    "notes": ""\n  },\n  {\n    "id": 57,\n    "question": "Qual número é ímpar: 14, 22, 31, 48?",\n    "expected_answer": "31",\n    "accepted_answers": [\n      "31"\n    ],\n    "category": "logic",\n    "notes": ""\n  },\n  {\n    "id": 58,\n    "question": "Se A implica B e A é verdadeiro, o que se conclui sobre B?",\n    "expected_answer": "B é verdadeiro",\n    "accepted_answers": [\n      "B é verdadeiro",\n      "verdadeiro",\n      "B true"\n    ],\n    "category": "logic",\n    "notes": ""\n  },\n  {\n    "id": 59,\n    "question": "Se uma receita pede 2 ovos para 1 bolo, quantos ovos são necessários para 4 bolos?",\n    "expected_answer": "8",\n    "accepted_answers": [\n      "8"\n    ],\n    "category": "logic",\n    "notes": ""\n  },\n  {\n    "id": 60,\n    "question": "Qual alternativa completa melhor: livro está para leitura assim como garfo está para ...?",\n    "expected_answer": "Comer",\n    "accepted_answers": [\n      "Comer",\n      "alimentação",\n      "eat"\n    ],\n    "category": "logic",\n    "notes": ""\n  },\n  {\n    "id": 61,\n    "question": "Em Python, qual palavra-chave define uma função?",\n    "expected_answer": "def",\n    "accepted_answers": [\n      "def"\n    ],\n    "category": "coding",\n    "notes": ""\n  },\n  {\n    "id": 62,\n    "question": "Qual estrutura em Python normalmente guarda pares chave-valor?",\n    "expected_answer": "dicionário",\n    "accepted_answers": [\n      "dicionário",\n      "dicionario",\n      "dict"\n    ],\n    "category": "coding",\n    "notes": ""\n  },\n  {\n    "id": 63,\n    "question": "Em JavaScript, qual valor booleano representa verdade?",\n    "expected_answer": "true",\n    "accepted_answers": [\n      "true"\n    ],\n    "category": "coding",\n    "notes": ""\n  },\n  {\n    "id": 64,\n    "question": "Qual comando Git cria um novo commit com as mudanças staged?",\n    "expected_answer": "git commit",\n    "accepted_answers": [\n      "git commit"\n    ],\n    "category": "coding",\n    "notes": ""\n  },\n  {\n    "id": 65,\n    "question": "Qual protocolo é usado normalmente para páginas web seguras?",\n    "expected_answer": "HTTPS",\n    "accepted_answers": [\n      "HTTPS"\n    ],\n    "category": "coding",\n    "notes": ""\n  },\n  {\n    "id": 66,\n    "question": "Qual comando Linux lista arquivos de um diretório?",\n    "expected_answer": "ls",\n    "accepted_answers": [\n      "ls"\n    ],\n    "category": "coding",\n    "notes": ""\n  },\n  {\n    "id": 67,\n    "question": "Em SQL, qual comando retorna linhas de uma tabela?",\n    "expected_answer": "SELECT",\n    "accepted_answers": [\n      "SELECT"\n    ],\n    "category": "coding",\n    "notes": ""\n  },\n  {\n    "id": 68,\n    "question": "Em HTML, qual tag principal representa um link?",\n    "expected_answer": "<a>",\n    "accepted_answers": [\n      "<a>",\n      "a"\n    ],\n    "category": "coding",\n    "notes": ""\n  },\n  {\n    "id": 69,\n    "question": "Em CSS, qual propriedade muda a cor do texto?",\n    "expected_answer": "color",\n    "accepted_answers": [\n      "color"\n    ],\n    "category": "coding",\n    "notes": ""\n  },\n  {\n    "id": 70,\n    "question": "Qual estrutura de dados usa a política LIFO?",\n    "expected_answer": "pilha",\n    "accepted_answers": [\n      "pilha",\n      "stack"\n    ],\n    "category": "coding",\n    "notes": ""\n  },\n  {\n    "id": 71,\n    "question": "Qual é o idioma oficial mais falado no Brasil?",\n    "expected_answer": "Português",\n    "accepted_answers": [\n      "Português",\n      "Portugues"\n    ],\n    "category": "language",\n    "notes": ""\n  },\n  {\n    "id": 72,\n    "question": "Qual é o antônimo de “rápido”?",\n    "expected_answer": "Lento",\n    "accepted_answers": [\n      "Lento"\n    ],\n    "category": "language",\n    "notes": ""\n  },\n  {\n    "id": 73,\n    "question": "Complete: cachorro está para latir assim como gato está para ...?",\n    "expected_answer": "miar",\n    "accepted_answers": [\n      "miar"\n    ],\n    "category": "language",\n    "notes": ""\n  },\n  {\n    "id": 74,\n    "question": "Qual é o plural de “papel” em português?",\n    "expected_answer": "papéis",\n    "accepted_answers": [\n      "papéis",\n      "papeis"\n    ],\n    "category": "language",\n    "notes": ""\n  },\n  {\n    "id": 75,\n    "question": "Qual palavra é um verbo: azul, correr, mesa, janela?",\n    "expected_answer": "correr",\n    "accepted_answers": [\n      "correr"\n    ],\n    "category": "language",\n    "notes": ""\n  },\n  {\n    "id": 76,\n    "question": "Qual é o sinônimo mais próximo de “feliz”?",\n    "expected_answer": "contente",\n    "accepted_answers": [\n      "contente",\n      "alegre"\n    ],\n    "category": "language",\n    "notes": ""\n  },\n  {\n    "id": 77,\n    "question": "Qual é a forma correta: “a gente fomos” ou “a gente foi”?",\n    "expected_answer": "a gente foi",\n    "accepted_answers": [\n      "a gente foi"\n    ],\n    "category": "language",\n    "notes": ""\n  },\n  {\n    "id": 78,\n    "question": "Em inglês, como se traduz “casa”?",\n    "expected_answer": "house",\n    "accepted_answers": [\n      "house"\n    ],\n    "category": "language",\n    "notes": ""\n  },\n  {\n    "id": 79,\n    "question": "Qual dessas palavras é um adjetivo: bonito, correr, pedra, ontem?",\n    "expected_answer": "bonito",\n    "accepted_answers": [\n      "bonito"\n    ],\n    "category": "language",\n    "notes": ""\n  },\n  {\n    "id": 80,\n    "question": "Qual palavra está escrita corretamente: exceção ou excessão?",\n    "expected_answer": "exceção",\n    "accepted_answers": [\n      "exceção",\n      "excecao"\n    ],\n    "category": "language",\n    "notes": ""\n  },\n  {\n    "id": 81,\n    "question": "Qual país venceu a Copa do Mundo de 2002?",\n    "expected_answer": "Brasil",\n    "accepted_answers": [\n      "Brasil",\n      "Brazil"\n    ],\n    "category": "sports",\n    "notes": ""\n  },\n  {\n    "id": 82,\n    "question": "Quantos jogadores cada time tem em campo no futebol tradicional?",\n    "expected_answer": "11",\n    "accepted_answers": [\n      "11",\n      "onze"\n    ],\n    "category": "sports",\n    "notes": ""\n  },\n  {\n    "id": 83,\n    "question": "Em que esporte se usa uma cesta e uma bola laranja?",\n    "expected_answer": "basquete",\n    "accepted_answers": [\n      "basquete",\n      "basketball"\n    ],\n    "category": "sports",\n    "notes": ""\n  },\n  {\n    "id": 84,\n    "question": "Em qual esporte Rafael Nadal ficou mundialmente famoso?",\n    "expected_answer": "tênis",\n    "accepted_answers": [\n      "tênis",\n      "tenis",\n      "tennis"\n    ],\n    "category": "sports",\n    "notes": ""\n  },\n  {\n    "id": 85,\n    "question": "Qual país sediou as Olimpíadas de 2016?",\n    "expected_answer": "Brasil",\n    "accepted_answers": [\n      "Brasil",\n      "Brazil"\n    ],\n    "category": "sports",\n    "notes": ""\n  },\n  {\n    "id": 86,\n    "question": "Qual piloto brasileiro é tricampeão mundial de Fórmula 1 e morreu em 1994?",\n    "expected_answer": "Ayrton Senna",\n    "accepted_answers": [\n      "Ayrton Senna",\n      "Senna"\n    ],\n    "category": "sports",\n    "notes": ""\n  },\n  {\n    "id": 87,\n    "question": "No xadrez, quantas casas tem um tabuleiro?",\n    "expected_answer": "64",\n    "accepted_answers": [\n      "64"\n    ],\n    "category": "sports",\n    "notes": ""\n  },\n  {\n    "id": 88,\n    "question": "Qual esporte usa raquete, rede e peteca?",\n    "expected_answer": "badminton",\n    "accepted_answers": [\n      "badminton"\n    ],\n    "category": "sports",\n    "notes": ""\n  },\n  {\n    "id": 89,\n    "question": "Em que esporte se faz touchdown?",\n    "expected_answer": "futebol americano",\n    "accepted_answers": [\n      "futebol americano",\n      "football americano",\n      "american football"\n    ],\n    "category": "sports",\n    "notes": ""\n  },\n  {\n    "id": 90,\n    "question": "Qual seleção é conhecida como La Albiceleste?",\n    "expected_answer": "Argentina",\n    "accepted_answers": [\n      "Argentina"\n    ],\n    "category": "sports",\n    "notes": ""\n  },\n  {\n    "id": 91,\n    "question": "Se você misturar azul com amarelo, qual cor obtém normalmente?",\n    "expected_answer": "verde",\n    "accepted_answers": [\n      "verde"\n    ],\n    "category": "everyday",\n    "notes": ""\n  },\n  {\n    "id": 92,\n    "question": "Qual eletrodoméstico é usado para manter alimentos congelados?",\n    "expected_answer": "freezer",\n    "accepted_answers": [\n      "freezer",\n      "congelador"\n    ],\n    "category": "everyday",\n    "notes": ""\n  },\n  {\n    "id": 93,\n    "question": "Quantos minutos tem 2 horas?",\n    "expected_answer": "120",\n    "accepted_answers": [\n      "120"\n    ],\n    "category": "everyday",\n    "notes": ""\n  },\n  {\n    "id": 94,\n    "question": "Qual objeto é usado para cortar papel com as mãos?",\n    "expected_answer": "tesoura",\n    "accepted_answers": [\n      "tesoura"\n    ],\n    "category": "everyday",\n    "notes": ""\n  },\n  {\n    "id": 95,\n    "question": "Qual é o mês que vem depois de abril?",\n    "expected_answer": "maio",\n    "accepted_answers": [\n      "maio"\n    ],\n    "category": "everyday",\n    "notes": ""\n  },\n  {\n    "id": 96,\n    "question": "Se um produto custa 50 reais e recebe desconto de 10 reais, quanto passa a custar?",\n    "expected_answer": "40",\n    "accepted_answers": [\n      "40",\n      "40 reais",\n      "R$40"\n    ],\n    "category": "everyday",\n    "notes": ""\n  },\n  {\n    "id": 97,\n    "question": "Qual bebida é tradicionalmente feita a partir de grãos de café torrados?",\n    "expected_answer": "café",\n    "accepted_answers": [\n      "café",\n      "cafe"\n    ],\n    "category": "everyday",\n    "notes": ""\n  },\n  {\n    "id": 98,\n    "question": "Qual direção é oposta ao norte?",\n    "expected_answer": "sul",\n    "accepted_answers": [\n      "sul",\n      "south"\n    ],\n    "category": "everyday",\n    "notes": ""\n  },\n  {\n    "id": 99,\n    "question": "Qual animal normalmente põe ovos e tem penas?",\n    "expected_answer": "ave",\n    "accepted_answers": [\n      "ave",\n      "pássaro",\n      "passaro",\n      "bird"\n    ],\n    "category": "everyday",\n    "notes": ""\n  },\n  {\n    "id": 100,\n    "question": "Quantos dias tem uma semana?",\n    "expected_answer": "7",\n    "accepted_answers": [\n      "7",\n      "sete"\n    ],\n    "category": "everyday",\n    "notes": ""\n  }\n]'
utils_payload = "import json\nimport re\nfrom collections import Counter\nfrom datetime import datetime\nfrom pathlib import Path\n\n\ndef load_questions(path):\n    path = Path(path)\n    with path.open('r', encoding='utf-8') as f:\n        return json.load(f)\n\n\ndef normalize_text(text):\n    text = '' if text is None else str(text)\n    text = text.strip().lower()\n    text = re.sub(r'\\s+', ' ', text)\n    text = text.replace('á', 'a').replace('à', 'a').replace('ã', 'a').replace('â', 'a')\n    text = text.replace('é', 'e').replace('ê', 'e')\n    text = text.replace('í', 'i')\n    text = text.replace('ó', 'o').replace('ô', 'o').replace('õ', 'o')\n    text = text.replace('ú', 'u')\n    text = text.replace('ç', 'c')\n    return text\n\n\ndef first_sentence(text):\n    text = (text or '').strip()\n    parts = re.split(r'(?<=[\\.!?])\\s+', text)\n    return parts[0] if parts and parts[0] else text\n\n\ndef answer_is_correct(prediction, accepted_answers):\n    pred = normalize_text(first_sentence(prediction))\n    accepted = [normalize_text(a) for a in accepted_answers]\n    if pred in accepted:\n        return True\n    return any(ans in pred for ans in accepted if ans)\n\n\ndef grade_predictions(questions, predictions):\n    by_id = {int(p['id']): p for p in predictions}\n    rows = []\n    for q in questions:\n        pred = by_id.get(int(q['id']), {})\n        answer = pred.get('model_answer', '')\n        correct = answer_is_correct(answer, q['accepted_answers'])\n        rows.append({\n            'id': q['id'],\n            'category': q['category'],\n            'question': q['question'],\n            'expected_answer': q['expected_answer'],\n            'model_answer': answer,\n            'is_correct': correct,\n            'raw_output': pred.get('raw_output', answer),\n        })\n    return rows\n\n\ndef summarize_rows(rows):\n    total = len(rows)\n    correct = sum(1 for r in rows if r['is_correct'])\n    by_cat = {}\n    counts = Counter(r['category'] for r in rows)\n    for category, n in counts.items():\n        cat_rows = [r for r in rows if r['category'] == category]\n        cat_correct = sum(1 for r in cat_rows if r['is_correct'])\n        by_cat[category] = {\n            'total': n,\n            'correct': cat_correct,\n            'accuracy': cat_correct / n if n else 0.0,\n        }\n    return {\n        'total_questions': total,\n        'correct': correct,\n        'accuracy': correct / total if total else 0.0,\n        'by_category': by_cat,\n    }\n\n\ndef save_json(data, out_path):\n    out_path = Path(out_path)\n    out_path.parent.mkdir(parents=True, exist_ok=True)\n    with out_path.open('w', encoding='utf-8') as f:\n        json.dump(data, f, ensure_ascii=False, indent=2)\n\n\ndef timestamped_result_path(results_dir, stem):\n    ts = datetime.now().strftime('%Y%m%d_%H%M%S')\n    return Path(results_dir) / f'{stem}_{ts}.json'\n"
(DATA_DIR / 'gemma4_100_questions.json').write_text(questions_payload, encoding='utf-8')
(UTILS_DIR / 'eval_utils.py').write_text(utils_payload, encoding='utf-8')

if str(UTILS_DIR) not in sys.path:
    sys.path.append(str(UTILS_DIR))

from eval_utils import load_questions, grade_predictions, summarize_rows, save_json, timestamped_result_path

DATA_PATH = DATA_DIR / 'gemma4_100_questions.json'
questions = load_questions(DATA_PATH)
print('PROJECT_ROOT =', PROJECT_ROOT)
print('Perguntas carregadas =', len(questions))


In [ ]:
!pip -q install peft bitsandbytes
import os
import gc
import re
from pathlib import Path
import torch
import torch.nn as nn
from huggingface_hub import login
from transformers import AutoTokenizer, AutoModelForCausalLM, TrainingArguments, Trainer, DataCollatorForLanguageModeling
from peft import LoraConfig, get_peft_model, AutoPeftModelForCausalLM
from datasets import Dataset

BASE_MODEL_NAME = 'google/gemma-2-2b-it'
PRUNED_CHECKPOINT = Path(CHECKPOINTS_DIR / 'student_pruned')
RECOVERY_OUTPUT = Path(CHECKPOINTS_DIR / 'student_ternary_recovered')
PROMPT_TEMPLATE = 'Responda de forma curta e direta.\nPergunta: {question}\nResposta:'
BATCH_SIZE = 1
MAX_NEW_TOKENS = 24
PILOT_LAYER_LIMIT = 8
PILOT_LAYER_NAME_FILTERS = ['mlp']
TRAIN_STEPS = 60
LEARNING_RATE = 5e-5
LORA_R = 8
LORA_ALPHA = 16
LORA_DROPOUT = 0.05

HF_TOKEN = os.environ.get('HF_TOKEN')
if IN_COLAB and not HF_TOKEN:
    try:
        from google.colab import userdata
        HF_TOKEN = userdata.get('HF_TOKEN')
    except Exception:
        HF_TOKEN = None

if BASE_MODEL_NAME.startswith('google/gemma') and not HF_TOKEN:
    raise RuntimeError('Gemma no Hugging Face é gated. Configure HF_TOKEN no Colab.')

if HF_TOKEN:
    login(token=HF_TOKEN, add_to_git_credential=False)


def cleanup_memory():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()


def checkpoint_has_weights(path: Path) -> bool:
    if not path.exists() or not path.is_dir():
        return False
    return any(path.glob('*.safetensors')) or any(path.glob('*.bin')) or any(path.glob('*.pt'))

if not checkpoint_has_weights(PRUNED_CHECKPOINT):
    raise FileNotFoundError(f'Checkpoint podado não encontrado em {PRUNED_CHECKPOINT}. Rode 03/04/05 antes.')


In [ ]:
def parse_answer(full_text):
    text = full_text.split('Resposta:')[-1].strip()
    text = re.sub(r'^[-: ]+', '', text)
    return text.strip()


def ask_batch(model, tokenizer, batch_questions, max_new_tokens=24):
    prompts = [PROMPT_TEMPLATE.format(question=q['question']) for q in batch_questions]
    inputs = tokenizer(prompts, return_tensors='pt', padding=True, truncation=True).to(model.device)
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.eos_token_id,
        )
    decoded = tokenizer.batch_decode(outputs, skip_special_tokens=True)
    rows = []
    for q, text in zip(batch_questions, decoded):
        rows.append({'id': q['id'], 'raw_output': text, 'model_answer': parse_answer(text)})
    return rows


def run_eval(model, tokenizer, questions, batch_size=1, max_new_tokens=24):
    predictions = []
    for start in range(0, len(questions), batch_size):
        batch = questions[start:start+batch_size]
        predictions.extend(ask_batch(model, tokenizer, batch, max_new_tokens=max_new_tokens))
    rows = grade_predictions(questions, predictions)
    summary = summarize_rows(rows)
    return predictions, rows, summary


def iter_linear_modules(model):
    for name, module in model.named_modules():
        if isinstance(module, nn.Linear):
            yield name, module


def choose_pilot_layers(model, name_filters=None, limit=8):
    chosen = []
    for name, module in iter_linear_modules(model):
        if name_filters and not any(f in name for f in name_filters):
            continue
        if 'lm_head' in name:
            continue
        chosen.append((name, module))
        if len(chosen) >= limit:
            break
    return chosen


def ternary_snap_(weight: torch.Tensor):
    scale = weight.detach().abs().mean().clamp(min=1e-8)
    q = torch.clamp(torch.round(weight / scale), -1, 1)
    weight.copy_(q * scale)
    return float(scale)


In [ ]:
tokenizer = AutoTokenizer.from_pretrained(PRUNED_CHECKPOINT, token=HF_TOKEN)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
teacher_model = AutoModelForCausalLM.from_pretrained(PRUNED_CHECKPOINT, dtype='auto', device_map='auto', token=HF_TOKEN)
teacher_model.eval()
student_model = AutoModelForCausalLM.from_pretrained(PRUNED_CHECKPOINT, dtype='auto', device_map='auto', token=HF_TOKEN)
student_model.eval()

pilot_layers = choose_pilot_layers(student_model, PILOT_LAYER_NAME_FILTERS, PILOT_LAYER_LIMIT)
pilot_layer_names = [name for name, _ in pilot_layers]
for name, module in student_model.named_modules():
    if name in pilot_layer_names and isinstance(module, nn.Linear):
        ternary_snap_(module.weight.data)

_, snapped_rows, snapped_summary = run_eval(student_model, tokenizer, questions, batch_size=BATCH_SIZE, max_new_tokens=MAX_NEW_TOKENS)
out_path = timestamped_result_path(RESULTS_DIR, '06_ternary_snapped_pre_recovery')
save_json({
    'source_model': str(PRUNED_CHECKPOINT),
    'pilot_layers': pilot_layer_names,
    'summary': snapped_summary,
    'rows': snapped_rows,
}, out_path)
print('Pre-recovery eval salvo em:', out_path)
print(snapped_summary)


In [ ]:
train_examples = []
for q in questions:
    prompt = PROMPT_TEMPLATE.format(question=q['question'])
    target = q['expected_answer']
    train_examples.append({'text': prompt + ' ' + target})

dataset = Dataset.from_list(train_examples)


def tokenize_row(row):
    enc = tokenizer(row['text'], truncation=True, max_length=128, padding='max_length')
    enc['labels'] = enc['input_ids'].copy()
    return enc

train_ds = dataset.map(tokenize_row, remove_columns=dataset.column_names)

lora_config = LoraConfig(
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj', 'up_proj', 'down_proj', 'gate_proj'],
    lora_dropout=LORA_DROPOUT,
    bias='none',
    task_type='CAUSAL_LM',
)

student_model = get_peft_model(student_model, lora_config)
student_model.print_trainable_parameters()


In [ ]:
class RecoveryTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        student_outputs = model(**inputs)
        student_logits = student_outputs.logits
        with torch.no_grad():
            teacher_outputs = teacher_model(input_ids=inputs['input_ids'], attention_mask=inputs.get('attention_mask'))
            teacher_logits = teacher_outputs.logits
        ce_loss = student_outputs.loss
        kl_loss = torch.nn.functional.kl_div(
            torch.nn.functional.log_softmax(student_logits, dim=-1),
            torch.nn.functional.softmax(teacher_logits, dim=-1),
            reduction='batchmean'
        )
        loss = 0.5 * ce_loss + 0.5 * kl_loss
        return (loss, student_outputs) if return_outputs else loss

args = TrainingArguments(
    output_dir=str(RECOVERY_OUTPUT),
    per_device_train_batch_size=1,
    gradient_accumulation_steps=8,
    learning_rate=LEARNING_RATE,
    max_steps=TRAIN_STEPS,
    logging_steps=10,
    save_steps=TRAIN_STEPS,
    bf16=torch.cuda.is_available(),
    report_to=[],
    remove_unused_columns=False,
)

trainer = RecoveryTrainer(
    model=student_model,
    args=args,
    train_dataset=train_ds,
    data_collator=DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False),
)
trainer.train()
trainer.save_model(str(RECOVERY_OUTPUT))
tokenizer.save_pretrained(str(RECOVERY_OUTPUT))
print('Recovered adapter salvo em:', RECOVERY_OUTPUT)


In [ ]:
# Libera VRAM: ao fim do treino, `teacher_model`, `student_model` e `trainer` ainda podem estar na GPU.
for _name in ['trainer', 'teacher_model', 'student_model']:
    if _name in globals():
        try:
            del globals()[_name]
        except Exception:
            pass

cleanup_memory()

# Em T4, evitar `merge_and_unload()` costuma poupar tempo/memória.
# Também usamos `dtype=` (API nova) em vez de `torch_dtype=`.
peft_model = AutoPeftModelForCausalLM.from_pretrained(
    str(RECOVERY_OUTPUT),
    token=HF_TOKEN,
    device_map='auto',
    dtype=torch.float16 if torch.cuda.is_available() else None,
    low_cpu_mem_usage=True,
)
peft_model.eval()

_, recovered_rows, recovered_summary = run_eval(
    peft_model,
    tokenizer,
    questions,
    batch_size=BATCH_SIZE,
    max_new_tokens=MAX_NEW_TOKENS
)

out_path = timestamped_result_path(RESULTS_DIR, '06_ternary_recovery_eval')
save_json({
    'source_model': str(PRUNED_CHECKPOINT),
    'recovered_model': str(RECOVERY_OUTPUT),
    'recovered_merged_model': None,
    'pilot_layers': pilot_layer_names,
    'summary': recovered_summary,
    'rows': recovered_rows,
}, out_path)

print('Recovery eval salvo em:', out_path)
print(recovered_summary)


In [ ]:
# Exporta artefatos e baixa automaticamente no Colab.
import shutil
from pathlib import Path

NOTEBOOK_STEM = Path(__file__).stem if '__file__' in globals() else 'notebook_run'
EXPORT_DIR = PROJECT_ROOT / 'exports'
EXPORT_DIR.mkdir(parents=True, exist_ok=True)

export_name = f"{NOTEBOOK_STEM}"
zip_base = EXPORT_DIR / export_name

bundle_paths = [RESULTS_DIR]
if CHECKPOINTS_DIR.exists() and any(CHECKPOINTS_DIR.iterdir()):
    bundle_paths.append(CHECKPOINTS_DIR)
if DATA_DIR.exists():
    bundle_paths.append(DATA_DIR)

staging_dir = EXPORT_DIR / f"{export_name}_bundle"
if staging_dir.exists():
    shutil.rmtree(staging_dir)
staging_dir.mkdir(parents=True, exist_ok=True)

for src in bundle_paths:
    dst = staging_dir / src.name
    if src.is_dir():
        shutil.copytree(src, dst, dirs_exist_ok=True)
    else:
        shutil.copy2(src, dst)

zip_path = shutil.make_archive(str(zip_base), 'zip', root_dir=staging_dir)
print('ZIP criado em:', zip_path)

if IN_COLAB:
    from google.colab import files
    files.download(zip_path)
